In [1]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report, confusion_matrix
import xgboost as xgb
import yaml
import os
import logging
from contextlib import nullcontext
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.metrics import ConfusionMatrixDisplay 
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder



logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

mlflow_tracking_uri = 'http://localhost:5555'


In [3]:
data.columns

Index(['text_length', 'word_count', 'tweet_year_2016', 'tweet_year_2017',
       'tweet_month_1', 'tweet_month_2', 'tweet_month_3', 'tweet_month_4',
       'tweet_month_5', 'tweet_month_6', 'tweet_month_7', 'tweet_month_8',
       'tweet_month_9', 'tweet_month_10', 'tweet_month_11', 'tweet_month_12',
       'tweet_day_1', 'tweet_day_2', 'tweet_day_3', 'tweet_day_4',
       'tweet_day_5', 'tweet_day_6', 'tweet_day_7', 'tweet_day_8',
       'tweet_day_9', 'tweet_day_10', 'tweet_day_11', 'tweet_day_12',
       'tweet_day_13', 'tweet_day_14', 'tweet_day_15', 'tweet_day_16',
       'tweet_day_17', 'tweet_day_18', 'tweet_day_19', 'tweet_day_20',
       'tweet_day_21', 'tweet_day_22', 'tweet_day_23', 'tweet_day_24',
       'tweet_day_25', 'tweet_day_26', 'tweet_day_27', 'tweet_day_28',
       'tweet_day_29', 'tweet_day_30', 'tweet_day_31', 'tweet_weekday_0',
       'tweet_weekday_1', 'tweet_weekday_2', 'tweet_weekday_3',
       'tweet_weekday_4', 'tweet_weekday_5', 'tweet_weekday_6', 'tweet_h

In [2]:
# Cell 2: Data Loading and Splitting
# Load dataset
data_path = '../data/processed/featured_house_data.csv'
data = pd.read_csv(data_path)

# Define features and target for classification
features = [
    'tweet_year',
    'tweet_month',
    'tweet_day',
    'tweet_weekday',
    'tweet_hour',
    'tweet_quarter',
    'text_length',
    'word_count'
]
target = 'sentiment'

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create encoder
label_encoder = LabelEncoder()

# Fit and transform y labels
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# Check mapping
label_mapping = {label: idx for idx, label in enumerate(label_encoder.classes_)}
print(label_mapping)


KeyError: "['tweet_year', 'tweet_month', 'tweet_day', 'tweet_weekday', 'tweet_hour', 'tweet_quarter'] not in index"

In [31]:
data

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,sentiment
0,700,115854,True,2017-10-31 22:16:56+00:00,@applesupport why are my i️’s changing not sho...,698,NaN,positive
1,714,115856,True,2017-10-31 22:19:32+00:00,hey @applesupport and anyone else who upgraded...,"712,715",NaN,neutral
2,723,115859,True,2017-10-31 22:11:16+00:00,@115858 @applesupport hello are all the lines ...,722,NaN,negative
3,730,115861,True,2017-10-31 20:46:35+00:00,"hello, internet. can someone explain why this ...","729,731",NaN,neutral
4,733,115863,True,2017-10-31 22:16:40+00:00,@applesupport i’ve got a screenshot saying my ...,732,NaN,neutral
...,...,...,...,...,...,...,...,...
50833,2987482,823733,True,2017-11-22 00:40:57+00:00,@applesupport why is my iphone 7 constantly se...,2987481,NaN,neutral
50834,2987605,689907,True,2017-11-22 02:11:43+00:00,hey @applesupport - not being able to duplicat...,2987604,NaN,negative
50835,2987607,823765,True,2017-11-22 02:17:14+00:00,yo @applesupport is that weird glitch w/ the c...,2987606,NaN,negative
50836,2987663,823779,True,2017-11-22 03:24:02+00:00,what the fuck @applesupport my phone keeps ha...,2987662,NaN,negative


In [ ]:
print("✅ Using  features for classification:")
for feature in features:
    print(f" - {feature}")
selected_features_dict = {
    'manual_selection': features
}

In [ ]:
#MLflow setup
if mlflow_tracking_uri:
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment("Sentiment Classification")

In [ ]:
models = {
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'XGBClassifier': xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', use_label_encoder=False, random_state=42) 
}

model_grids = {
    'GradientBoostingClassifier': {
        'n_estimators': [10],
        'learning_rate': [0.1],
        'max_depth': [3]
    },
    'XGBClassifier': {
        'n_estimators': [90],
        'learning_rate': [0.1],
        'max_depth': [3]
    }
}

In [ ]:
def evaluate_model_with_gridsearch(name, model, grid, X_train, y_train, X_test, y_test):
    if grid:
    
        clf = GridSearchCV(model, grid, cv=3, scoring='f1_weighted', n_jobs=-1) # Changed scoring to 'f1_weighted'
        clf.fit(X_train, y_train)
        best_model = clf.best_estimator_
        best_params = clf.best_params_
    else:
        model.fit(X_train, y_train)
        best_model = model
        best_params = model.get_params()

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test) if hasattr(best_model, 'predict_proba') else None

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted') # Use weighted for multi-class
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    
    roc_auc = None
    if y_proba is not None:
        if len(np.unique(y_test)) == 2: # Binary classification
            roc_auc = roc_auc_score(y_test, y_proba[:, 1])
        elif len(np.unique(y_test)) > 2: # Multi-class classification
            try:
                roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
            except ValueError:
                logger.warning("ROC AUC score cannot be computed for multi-class classification with a single class in y_true or y_score.")
                roc_auc = None

    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'roc_auc': roc_auc, # Included ROC AUC
        'model': best_model,
        'params': best_params,
        'y_pred': y_pred,
        'y_test': y_test # Return y_test and y_pred for classification report/confusion matrix
    }

print("MLflow tracking URI:", mlflow_tracking_uri)

In [ ]:
results = {}

with mlflow.start_run(run_name="classification_model_comparison") if mlflow_tracking_uri else nullcontext(): # Updated run name
    for name, model in models.items():
        logger.info(f"Training {name}...")
        with mlflow.start_run(run_name=name, nested=True) if mlflow_tracking_uri else nullcontext():
            evaluation = evaluate_model_with_gridsearch(name, model, model_grids[name], X_train, y_train, X_test, y_test)
            results[name] = evaluation

            if mlflow_tracking_uri:
                mlflow.log_params(evaluation['params'])
                metrics_to_log = {
                    'accuracy': evaluation['accuracy'],
                    'f1_score': evaluation['f1_score'],
                    'precision': evaluation['precision'],
                    'recall': evaluation['recall']
                }
                if evaluation['roc_auc'] is not None:
                    metrics_to_log['roc_auc'] = evaluation['roc_auc']
                mlflow.log_metrics(metrics_to_log) # Log classification metrics
                mlflow.sklearn.log_model(evaluation['model'], artifact_path=name.lower().replace(" ", "_"))
            
            print(f"{name} Accuracy: {evaluation['accuracy']:.4f}, F1-score (weighted): {evaluation['f1_score']:.4f}") 

            print(f"\nClassification Report for {name}:\n")
            print(classification_report(evaluation['y_test'], evaluation['y_pred']))

            fig, ax = plt.subplots(figsize=(8, 6))
            cmp = ConfusionMatrixDisplay.from_predictions(evaluation['y_test'], evaluation['y_pred'], ax=ax, cmap=plt.cm.Blues)
            ax.set_title(f"Confusion Matrix for {name}")
            plt.show()
            if mlflow_tracking_uri:
                mlflow.log_figure(fig, f"{name.lower().replace(' ', '_')}_confusion_matrix.png") # Log confusion matrix plot

In [ ]:
best_model_name = max(results, key=lambda k: results[k]['f1_score']) 
best_model = results[best_model_name]['model']
best_params = results[best_model_name]['params']
best_accuracy = float(results[best_model_name]['accuracy'])
best_f1_score = float(results[best_model_name]['f1_score'])
best_precision = float(results[best_model_name]['precision'])
best_recall = float(results[best_model_name]['recall'])
best_roc_auc = float(results[best_model_name]['roc_auc']) if results[best_model_name]['roc_auc'] is not None else 'N/A'

print(f"🏆 Best Model: {best_model_name}")
print(f"   Accuracy: {best_accuracy:.4f}")
print(f"   F1-score (weighted): {best_f1_score:.4f}")
print(f"   Precision (weighted): {best_precision:.4f}")
print(f"   Recall (weighted): {best_recall:.4f}")
if best_roc_auc != 'N/A':
    print(f"   ROC AUC (weighted/ovr): {best_roc_auc:.4f}")

# Save the best model using joblib with a classification-specific name
model_filename = f'../models/{best_model_name.lower().replace(" ", "_")}_sentiment_classifier.joblib'
os.makedirs(os.path.dirname(model_filename), exist_ok=True)
joblib.dump(best_model, model_filename)
print(f"\nBest model saved to {model_filename}")

# Create model configuration for classification
model_config = {
    'model': {
        'name': 'sentiment_classification_model', # Updated model name
        'best_model': best_model_name,
        'parameters': best_params,
        'accuracy_score': best_accuracy, # Stored classification metrics
        'f1_score': best_f1_score,
        'precision_score': best_precision,
        'recall_score': best_recall,
        'roc_auc_score': best_roc_auc,
        'target_variable': target, # Updated target variable
        'feature_sets': selected_features_dict # Storing the selected features
    }
}

# Save model configuration with a classification-specific name
config_path = '../configs/model_config_classification.yaml' # Changed config file name
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)
print(f"Model configuration saved to {config_path}")

# Optional: Log the best model and config to MLflow as well
if mlflow_tracking_uri:
    with mlflow.start_run(run_name="best_model_final_log") if mlflow_tracking_uri else nullcontext():
        mlflow.log_params(best_params)
        final_metrics = {
            'best_accuracy': best_accuracy,
            'best_f1_score': best_f1_score,
            'best_precision': best_precision,
            'best_recall': best_recall
        }
        if best_roc_auc != 'N/A':
            final_metrics['best_roc_auc'] = best_roc_auc
        mlflow.log_metrics(final_metrics)
        mlflow.sklearn.log_model(best_model, artifact_path="best_sentiment_classifier")
        mlflow.log_artifact(config_path) # Log the configuration file
        print("Final best model and configuration logged to MLflow.")
